# 01 — Survey cleaning and scoring

**Input:** `data/raw/survey_raw.xlsx` (restricted). **Output:** processed survey data for Notebooks 02–05.

Run from the project root or `notebooks/` using the project Python environment. This notebook checks the export, scores the GAD-7 and AI-use variables, and calculates descriptive and reliability statistics. Participant-level files and notebook outputs are **not for public release**.

In [ ]:
# Import libraries
# This step loads the Python tools needed for the analysis.
# pandas handles the survey dataset, while numpy provides
# additional numerical functionality.

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

In [ ]:
# Define project paths
# The notebook is stored inside /notebooks, while the data
# folders are one level above it. Defining the paths here keeps
# file locations clear and avoids repeating them later.

from pathlib import Path

# Locate the project root from the repository or any notebook subfolder.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / "data" / "raw" / "survey_raw.xlsx").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate data/raw/survey_raw.xlsx from {cwd}. "
        "Check that the raw dataset exists in the thesis project."
    )

# Define project directories.
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "survey_raw.xlsx"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
TABLES = PROJECT_ROOT / "tables"

# Confirm the correct paths.
print("Project root:", PROJECT_ROOT)
print("Raw dataset:", RAW_DATA)
print("Raw dataset found:", RAW_DATA.is_file())

In [ ]:
# Check that the raw data exists
# Before attempting to load the survey, this check confirms
# that Python can find the file at the expected location.

if RAW_DATA.exists():
    print("✓ Raw survey file found.")
else:
    print("✗ Raw survey file NOT found.")
    print("Expected location:", RAW_DATA)

In [ ]:
# Load the raw survey
# The original Excel export is loaded into a pandas DataFrame.
# The raw Excel file itself is not changed.

df_raw = pd.read_excel(RAW_DATA)

print("Survey loaded successfully.")
print("Number of responses:", df_raw.shape[0])
print("Number of variables:", df_raw.shape[1])

In [ ]:
# Preview the data
# Looking at the first few responses helps verify that the
# Excel file was imported correctly before any transformations
# are made.

df_raw.head()

In [ ]:
# Inspect column names
# LimeSurvey exports can contain long or awkward variable
# names. Before renaming or scoring anything, this step shows
# exactly what Python received.

for i, column in enumerate(df_raw.columns):
    print(f"{i}: {column}")

In [ ]:
# Create a working copy
# The raw import is kept untouched in df_raw. All cleaning and
# scoring will instead be performed on df.

df = df_raw.copy()

print("Working copy created.")
print("Shape:", df.shape)

In [ ]:
# Check missing values
# Completed questionnaires can still contain missing values
# when individual questions were optional. This step therefore
# checks missingness rather than assuming completeness.

missing_values = df.isna().sum()

missing_values = missing_values[missing_values > 0].sort_values(
    ascending=False
)

print("Variables containing missing values:")
display(missing_values)

In [ ]:
# Check duplicate rows
# Duplicate responses could distort descriptive statistics and
# correlations, so they are identified before analysis.

duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

In [ ]:
# Check current column names
# This displays every column name currently present in the
# dataframe. The output will be used to identify the exact
# survey columns corresponding to the GAD-7 and AI
# self-efficacy questionnaire items.

for i, column in enumerate(df.columns):
    print(i, repr(column))

In [ ]:
# Rename analysis variables
# The LimeSurvey export contains long column names consisting
# of a question code followed by the complete question text.
# For the analysis, shorter and more descriptive variable names
# make the dataset easier to work with and the later statistical
# code easier to understand.
#
# The main quantitative variables are organised into three
# groups:
#
#   gad1-gad7       = GAD-7 anxiety items
#   aise1-aise6     = AI self-efficacy items
#   llm_*           = frequency of LLM use in coding tasks
#
# The general frequency of LLM use for coding-related tasks is
# stored separately as "llm_overall".
#
# This step changes only the column names. It does not change
# any participant responses.
#
# The code is written so that it can safely be run again. If a
# variable has already been renamed, it will simply be skipped.


# Define the survey codes and their new analysis names

variable_codes = {

    # GAD-7 anxiety items
    "G02Q06[SQ001]": "gad1",
    "G02Q06[SQ002]": "gad2",
    "G02Q06[SQ003]": "gad3",
    "G02Q06[SQ004]": "gad4",
    "G02Q06[SQ005]": "gad5",
    "G02Q06[SQ006]": "gad6",
    "G02Q06[SQ007]": "gad7",

    # AI self-efficacy items
    "G02Q07[SQ001]": "aise1",
    "G02Q07[SQ002]": "aise2",
    "G02Q07[SQ003]": "aise3",
    "G02Q07[SQ004]": "aise4",
    "G02Q07[SQ005]": "aise5",
    "G02Q07[SQ006]": "aise6",

    # Overall LLM-use frequency
    "G03Q08": "llm_overall",

    # LLM use for specific coding tasks
    "G03Q09[SQ001]": "llm_debug",
    "G03Q09[SQ004]": "llm_errors",
    "G03Q09[SQ003]": "llm_generate",
    "G03Q09[SQ002]": "llm_refactor",
    "G03Q09[SQ005]": "llm_validate",
}


# Build the renaming map
# For each variable, first check whether the shorter analysis
# name already exists. This is useful in Jupyter notebooks,
# where a cell may be run more than once during development.

rename_map = {}

for survey_code, new_name in variable_codes.items():

    # If the column has already been renamed, no further action
    # is needed for that variable.
    if new_name in df.columns:
        continue

    # Otherwise, find the original LimeSurvey column whose name
    # begins with the corresponding question code.
    matches = [
        column
        for column in df.columns
        if column.startswith(survey_code)
    ]

    # Each survey code should correspond to exactly one column.
    if len(matches) == 0:
        raise ValueError(
            f"No column found for survey code: {survey_code}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"More than one column found for survey code "
            f"{survey_code}: {matches}"
        )

    rename_map[matches[0]] = new_name


# Apply the new variable names

df = df.rename(columns=rename_map)

print("Analysis variables successfully renamed.")

In [ ]:
# Verify renamed analysis variables
# Before scoring or transforming questionnaire responses, this
# step checks that all variables needed for the main quantitative
# analysis were successfully renamed.
#
# These variables represent the three central quantitative
# components of the study:
#
#   1. GAD-7 anxiety symptoms
#   2. AI self-efficacy
#   3. LLM use in coding-related tasks

analysis_columns = [

    # GAD-7
    "gad1",
    "gad2",
    "gad3",
    "gad4",
    "gad5",
    "gad6",
    "gad7",

    # AI self-efficacy
    "aise1",
    "aise2",
    "aise3",
    "aise4",
    "aise5",
    "aise6",

    # LLM use
    "llm_overall",
    "llm_debug",
    "llm_errors",
    "llm_generate",
    "llm_refactor",
    "llm_validate",
]

print(df[analysis_columns].head())

In [ ]:
# Inspect questionnaire response categories
# Before converting questionnaire responses into numerical
# scores, this step checks the exact response categories
# currently present in each analysis variable.
#
# This provides a final validation that the scoring rules below
# correspond to the actual survey responses and helps detect
# unexpected values before any transformation is performed.

for column in analysis_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))

In [ ]:
# Score gad-7 items
# The GAD-7 consists of seven items measuring the frequency of
# anxiety symptoms during the previous two weeks.
#
# Responses are converted according to the standard GAD-7
# scoring system:
#
#   Not at all                 = 0
#   Several days               = 1
#   More than half the days    = 2
#   Nearly every day           = 3
#
# Higher values indicate more frequent anxiety symptoms.
#
# replace() is used so that this cell can safely be rerun if
# the responses have already been converted to numbers.

gad_columns = [
    "gad1",
    "gad2",
    "gad3",
    "gad4",
    "gad5",
    "gad6",
    "gad7"
]

gad_mapping = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3
}

for column in gad_columns:
    df[column] = df[column].replace(gad_mapping)
    df[column] = pd.to_numeric(df[column], errors="raise")

In [ ]:
# Score ai self-efficacy items
# The six AI self-efficacy items use a seven-point Likert scale
# ranging from completely disagree to completely agree.
#
# Responses are converted as follows:
#
#   Completely disagree = 1
#   Mostly disagree     = 2
#   Slightly disagree   = 3
#   Neutral             = 4
#   Slightly agree      = 5
#   Mostly agree        = 6
#   Completely agree    = 7
#
# Higher values indicate greater AI self-efficacy.
#
# All six items are positively oriented, so no reverse scoring
# is required.

aise_columns = [
    "aise1",
    "aise2",
    "aise3",
    "aise4",
    "aise5",
    "aise6"
]

aise_mapping = {
    "Completely disagree": 1,
    "Mostly disagree": 2,
    "Slightly disagree": 3,
    "Neutral": 4,
    "Slightly agree": 5,
    "Mostly agree": 6,
    "Completely agree": 7
}

for column in aise_columns:
    df[column] = df[column].replace(aise_mapping)
    df[column] = pd.to_numeric(df[column], errors="raise")

In [ ]:
# Score llm-use frequency items
# LLM-use frequency is measured using a five-point response
# scale:
#
#   Never        = 1
#   Rarely       = 2
#   Sometimes    = 3
#   Often        = 4
#   Always       = 5
#
# Higher values therefore indicate more frequent use of LLMs
# for coding-related activities.
#
# The overall LLM-use item is kept separate from the five
# task-specific items so that general and task-specific usage
# patterns can both be examined later.

llm_columns = [
    "llm_overall",
    "llm_debug",
    "llm_errors",
    "llm_generate",
    "llm_refactor",
    "llm_validate"
]

llm_task_columns = [
    "llm_debug",
    "llm_errors",
    "llm_generate",
    "llm_refactor",
    "llm_validate"
]

llm_mapping = {
    "Never": 1,
    "Rarely": 2,
    "Sometimes": 3,
    "Often": 4,
    "Always": 5
}

for column in llm_columns:
    df[column] = df[column].replace(llm_mapping)
    df[column] = pd.to_numeric(df[column], errors="raise")

In [ ]:
# Verify numerical scoring
# After converting the questionnaire responses into numerical
# values, this step checks that all observed scores fall within
# their expected ranges.
#
# Expected item ranges:
#
#   GAD-7 items          = 0 to 3
#   AI self-efficacy     = 1 to 7
#   LLM-use frequency    = 1 to 5
#
# Missing values are also checked before participant-level
# summary scores are calculated.

print("GAD-7 item ranges:")
print(df[gad_columns].agg(["min", "max"]))

print("\nAI self-efficacy item ranges:")
print(df[aise_columns].agg(["min", "max"]))

print("\nLLM-use item ranges:")
print(df[llm_columns].agg(["min", "max"]))

print("\nMissing values:")
print(df[analysis_columns].isna().sum())

In [ ]:
# Calculate participant-level scale scores
# After confirming that all questionnaire items were scored
# correctly, this step combines the individual items into
# participant-level summary measures.
#
# These summary variables represent the main constructs used
# in the later statistical analyses.
#
# GAD-7:
# The seven items are summed, producing a total score ranging
# from 0 to 21. Higher scores indicate more frequent anxiety
# symptoms.
#
# AI self-efficacy:
# The mean of the six items is calculated. Using the mean keeps
# the resulting score on the original 1-to-7 response scale,
# which makes the score easier to interpret.
#
# LLM task frequency:
# The mean of the five task-specific LLM-use items is
# calculated. This creates a general indicator of how
# frequently each participant uses LLMs across coding-related
# tasks.
#
# The individual item variables are kept in the dataframe so
# that reliability checks and task-specific analyses can still
# be performed later.


# GAD-7 total score

df["gad7_total"] = df[gad_columns].sum(axis=1, min_count=len(gad_columns))


# AI self-efficacy score

df["aise_mean"] = df[aise_columns].mean(axis=1)


# A total score is also retained in case it is useful for
# descriptive reporting or comparison with previous research.

df["aise_total"] = df[aise_columns].sum(axis=1)


# LLM task-frequency score

df["llm_task_mean"] = df[llm_task_columns].mean(axis=1)

In [ ]:
# Verify participant-level scores
# This step checks the newly calculated summary variables before
# they are used in any statistical analysis.
#
# Expected theoretical ranges:
#
#   GAD-7 total          = 0 to 21
#   AI self-efficacy mean = 1 to 7
#   AI self-efficacy total = 6 to 42
#   Overall LLM use      = 1 to 5
#   LLM task mean        = 1 to 5
#
# The observed minimum and maximum do not have to equal the
# theoretical limits, because the sample may simply not contain
# participants at the extreme ends of the scales.

score_columns = [
    "gad7_total",
    "aise_mean",
    "aise_total",
    "llm_overall",
    "llm_task_mean"
]

print("First five participant-level scores:")
print(df[score_columns].head())

print("\nSummary of participant-level scores:")
print(
    df[score_columns].agg(
        ["min", "max", "mean", "std"]
    )
)

print("\nMissing values in participant-level scores:")
print(df[score_columns].isna().sum())

In [ ]:
# Check internal consistency with cronbach's alpha
# Cronbach's alpha evaluates the internal consistency of a set
# of items: that is, the extent to which responses to the items
# tend to vary together.
#
# In this study, alpha is calculated for:
#
#   1. GAD-7
#   2. AI self-efficacy
#   3. The five task-specific LLM-use items
#
# An "alpha if item deleted" diagnostic is also calculated.
# This shows how the overall alpha would change if each item
# were omitted from the calculation.
#
# IMPORTANT:
# These diagnostics are exploratory and do not provide an
# automatic justification for deleting questionnaire items.
#
# For established or adapted measures, decisions about item
# retention must also consider the theoretical structure and
# original instrument. A small increase in alpha after removing
# an item does not by itself mean that the item should be
# removed.
#
# This is especially important for the LLM task-frequency
# variables, because the items describe different coding
# behaviours rather than necessarily interchangeable indicators
# of a single psychological construct.


# Function for calculating Cronbach's alpha

def cronbach_alpha(data):
    """
    Calculate Cronbach's alpha for a set of questionnaire items.

    Parameters
    ----------
    data : pandas DataFrame
        Dataframe containing only the items belonging to the
        measure being evaluated.

    Returns
    -------
    float
        Cronbach's alpha.
    """

    data = data.dropna()

    number_of_items = data.shape[1]

    item_variances = data.var(
        axis=0,
        ddof=1
    )

    total_scores = data.sum(axis=1)

    total_variance = total_scores.var(
        ddof=1
    )

    alpha = (
        number_of_items
        / (number_of_items - 1)
    ) * (
        1
        - item_variances.sum()
        / total_variance
    )

    return alpha


# Function for reliability diagnostics

def evaluate_scale_reliability(data, scale_name):
    """
    Calculate overall Cronbach's alpha and alpha-if-item-deleted
    values for a multi-item measure.

    The item-deletion results are diagnostic only and should not
    be interpreted as automatic recommendations to remove items.
    """

    data = data.dropna()

    overall_alpha = cronbach_alpha(data)

    print(
        "============================================================"
    )
    print(
        f"RELIABILITY ANALYSIS: {scale_name}"
    )
    print(
        "============================================================"
    )

    print(
        f"Overall Cronbach's alpha: "
        f"{overall_alpha:.3f}"
    )

    print(
        "\nAlpha if item deleted:"
    )

    diagnostic_results = []

    for column in data.columns:

        remaining_columns = [
            c for c in data.columns
            if c != column
        ]

        alpha_deleted = cronbach_alpha(
            data[remaining_columns]
        )

        difference = (
            alpha_deleted - overall_alpha
        )

        diagnostic_results.append({
            "Item": column,
            "Alpha if deleted": alpha_deleted,
            "Change in alpha": difference
        })


    # Convert the diagnostic results into a clean table.
    reliability_table = pd.DataFrame(
        diagnostic_results
    )

    reliability_table[
        "Alpha if deleted"
    ] = reliability_table[
        "Alpha if deleted"
    ].round(3)

    reliability_table[
        "Change in alpha"
    ] = reliability_table[
        "Change in alpha"
    ].round(3)

    display(reliability_table)

    print(
        "\nInterpretation note:"
    )

    print(
        "A positive change means that alpha would increase if "
        "the item were omitted; a negative change means that "
        "alpha would decrease. These values are descriptive "
        "diagnostics and are not automatic criteria for item "
        "removal."
    )

    print("\n")


# Run reliability evaluations

evaluate_scale_reliability(
    df[gad_columns],
    "GAD-7 Anxiety Scale"
)

evaluate_scale_reliability(
    df[aise_columns],
    "AI Self-Efficacy Scale"
)

evaluate_scale_reliability(
    df[llm_task_columns],
    "LLM Task-Frequency Items"
)

In [ ]:
# Descriptive statistics for core variables
# This step summarizes the main participant-level variables used
# in the quantitative analysis.
#
# The descriptive statistics provide information about the
# central tendency, variability, and observed range of:
#
#   - anxiety symptoms (GAD-7)
#   - AI self-efficacy
#   - overall LLM-use frequency
#   - task-specific LLM-use frequency
#
# These results provide an initial overview of the sample before
# relationships between variables are investigated.

core_scores = [
    "gad7_total",
    "aise_mean",
    "llm_overall",
    "llm_task_mean"
]

descriptive_statistics = df[core_scores].describe().T

print(descriptive_statistics)

In [ ]:
# Create gad-7 severity categories
# GAD-7 total scores are grouped into standard clinical symptom
# severity categories for descriptive reporting (Spitzer et al., 2006).
#
# Score Cutoffs:
#   0-4   = Minimal
#   5-9   = Mild
#   10-14 = Moderate
#   15-21 = Severe
#
# Parameter choices:
#   - bins=[-1, 4, 9, 14, 21] with right=True creates intervals:
#     (-1, 4]   --> scores 0, 1, 2, 3, 4
#     (4, 9]    --> scores 5, 6, 7, 8, 9
#     (9, 14]   --> scores 10, 11, 12, 13, 14
#     (14, 21]  --> scores 15, 16, 17, 18, 19, 20, 21
#   - ordered=True ensures pandas treats severity as an ordinal
#     variable, preserving correct ordering in plots and tables.
#   - Range labels (0-4, 5-9, etc.) are embedded directly in category
#     names so that figures and summary tables are self-explanatory.

df["gad7_severity"] = pd.cut(
    df["gad7_total"],
    bins=[-1, 4, 9, 14, 21],
    right=True,
    ordered=True,
    labels=[
        "Minimal (0-4)",
        "Mild (5-9)",
        "Moderate (10-14)",
        "Severe (15-21)"
    ]
)

# Generate a clean summary table combining counts and percentages

severity_counts = df["gad7_severity"].value_counts().sort_index()
severity_pcts = (
    df["gad7_severity"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(1)
)

gad7_summary_table = pd.DataFrame({
    "Count (N)": severity_counts,
    "Percentage (%)": severity_pcts
})

print("============================================================")
print("             GAD-7 ANXIETY SEVERITY DISTRIBUTION             ")
print("============================================================")
display(gad7_summary_table)

In [ ]:
# Save the processed dataset
# The cleaned, renamed, numerically scored, and derived
# variables are saved as a separate processed dataset.
#
# The original raw survey export remains unchanged. Keeping raw
# and processed data separate makes the analysis reproducible
# and allows the complete cleaning procedure to be rerun from
# the original source if necessary.

from pathlib import Path

# Determine the project root depending on where the notebook is
# currently being executed.
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

# Create the processed-data directory if it does not yet exist.
processed_directory = project_root / "data" / "processed"
processed_directory.mkdir(
    parents=True,
    exist_ok=True
)

# Define the output file.
processed_file = (
    processed_directory / "survey_processed.csv"
)

# Save the processed dataframe.
df.to_csv(
    processed_file,
    index=False
)

print("Processed dataset saved successfully:")
print(processed_file)

In [ ]:
# Final cleaning summary
# This final check summarizes the processed dataset and confirms
# that the main analysis variables are available before the
# statistical analysis is continued in a separate notebook.

print("Number of participants:")
print(len(df))

print("\nNumber of variables:")
print(df.shape[1])

print("\nCore analysis variables:")
print([
    "gad7_total",
    "aise_mean",
    "llm_overall",
    "llm_task_mean"
])

print("\nMissing values in core analysis variables:")
print(
    df[
        [
            "gad7_total",
            "aise_mean",
            "llm_overall",
            "llm_task_mean"
        ]
    ].isna().sum()
)

print("\nCleaning and scoring completed successfully.")

In [ ]:
print("=== NOTEBOOK 01 VALIDATION ===")

print("Project root:", PROJECT_ROOT)

print("\nAvailable dataframes:")
for name, value in globals().copy().items():
    if isinstance(value, pd.DataFrame):
        print(
            f"{name}: "
            f"{value.shape[0]} rows, "
            f"{value.shape[1]} columns"
        )

In [ ]:
print("=== FINAL CLEANING VALIDATION ===")

# 1. Dataset dimensions
print("\nOriginal shape:", df_raw.shape)
print("Cleaned shape:", df.shape)

# 2. Missing values
print("\nMissing values per cleaned column:")
print(df.isna().sum().to_string())

# 3. Numeric descriptive statistics
print("\nDescriptive statistics:")
print(descriptive_statistics.to_string(index=False))

# 4. GAD-7 summary
print("\nGAD-7 summary:")
print(gad7_summary_table.to_string(index=False))

In [ ]:
print("=== THESIS VALIDATION ===")

print("\nDataset:")
print("Original:", df_raw.shape)
print("Cleaned:", df.shape)

print("\nDescriptive statistics:")
display(descriptive_statistics)

print("\nGAD-7 summary:")
display(gad7_summary_table)

print("\nMissing values in derived variables:")
derived_columns = [
    col for col in df.columns
    if col not in df_raw.columns
]

for col in derived_columns:
    print(f"{col}: {df[col].isna().sum()} missing")

In [ ]:
print(descriptive_statistics.to_string(index=False))